In [ ]:
!pip install faiss-cpu

In [ ]:
!pip install -q langchain-mistralai

In [ ]:
import pandas as pd
import faiss
import numpy as np
import requests
from google.colab import userdata

# Extracción de la clave y configuración de las cabeceras para la API
api_key = userdata.get('MISTRAL_API_KEY')
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

print("✅ Entorno base y credenciales configurados.")

✅ Entorno base y credenciales configurados.


In [ ]:
# 1. Cargar el dataset
df = pd.read_csv('healthcare_dataset.csv')

# 2. Limpieza básica para evitar errores con valores nulos
df = df.fillna("Desconocido")

# 3. Preparación del contexto (transformar filas en lenguaje natural)
# Puedes ajustar esta línea para incluir las columnas que consideres más relevantes
textos = df.apply(lambda row: f"Paciente: {row['Name']}. Edad: {row['Age']}. Condición médica: {row['Medical Condition']}. Tipo de sangre: {row['Blood Type']}.", axis=1).tolist()

print(f"✅ CSV preparado. {len(textos)} registros normalizados y listos para procesar.")

✅ CSV preparado. 55500 registros normalizados y listos para procesar.


In [ ]:
import os
from google.colab import userdata
from langchain_mistralai import MistralAIEmbeddings

# Configurar la API Key si no lo habías hecho en esta sesión
if "MISTRAL_API_KEY" not in os.environ:
    os.environ["MISTRAL_API_KEY"] = userdata.get('MISTRAL_API_KEY')

# Inicializar el modelo de embeddings de Mistral
embeddings = MistralAIEmbeddings(model="mistral-embed")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

In [ ]:
import time
import os
import pickle
import numpy as np

def get_embeddings_en_lotes_con_retry(texts, batch_size=100, checkpoint_file="embeddings_checkpoint.pkl"):
    all_embeddings = []
    inicio_lote = 0

    # 1. Intentar reanudar desde un checkpoint si existe
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, 'rb') as f:
            checkpoint_data = pickle.load(f)
            all_embeddings = checkpoint_data.get('embeddings', [])
            inicio_lote = checkpoint_data.get('ultimo_lote', 0)
        print(f"🔄 Checkpoint detectado. Reanudando desde el lote {inicio_lote} ({len(all_embeddings)} textos ya procesados).")

    total_textos = len(texts)

    for i in range(inicio_lote, total_textos, batch_size):
        batch = texts[i:i + batch_size]
        retries = 5
        espera = 4  # Tiempo inicial de espera en segundos

        while retries > 0:
            try:
                # Modifica esta línea según cómo llames exactamente a los embeddings de Mistral en tu celda
                # Ejemplo asumiendo uso de client.embeddings.create o langchain:
                # response = client.embeddings.create(model="mistral-embed", input=batch)

                # --- AQUÍ VA TU LLAMADA ACTUAL A LA API ---
                # Reemplaza la línea de abajo por tu lógica de extracción actual:
                batch_embeddings = [embeddings.embed_query(t) for t in batch]
                # ------------------------------------------

                all_embeddings.extend(batch_embeddings)
                print(f"✅ Lote {i} a {min(i + batch_size, total_textos)} procesado correctamente.")

                # Guardar checkpoint al finalizar con éxito el lote
                with open(checkpoint_file, 'wb') as f:
                    pickle.dump({'embeddings': all_embeddings, 'ultimo_lote': i + batch_size}, f)

                break  # Éxito, salimos del bucle de reintentos

            except Exception as e:
                # Si el error es por Rate Limit (429) o similar, aplicamos backoff
                if "Rate limit" in str(e) or "429" in str(e):
                    print(f"\n⚠️ Rate limit alcanzado en lote {i}. Reintentando en {espera} segundos... (Intentos restantes: {retries})")
                    time.sleep(espera)
                    espera *= 2  # Duplicamos el tiempo de espera para el siguiente intento
                    retries -= 1
                else:
                    # Si es otro tipo de error crítico, guardamos lo que tenemos y frenamos
                    with open(checkpoint_file, 'wb') as f:
                        pickle.dump({'embeddings': all_embeddings, 'ultimo_lote': i}, f)
                    raise e

        if retries == 0:
            print(f"\n❌ Se agotaron los reintentos en el lote {i}. El progreso se guardó en {checkpoint_file}.")
            raise ValueError("La API de Mistral sigue rechazando la petición por Rate Limit tras múltiples intentos.")

        # Un pequeño respiro de cortesía entre lotes normales para no saturar la API
        time.sleep(1)

    # Si todo termina con éxito, limpiamos el archivo de checkpoint opcionalmente
    if os.path.exists(checkpoint_file):
        os.remove(checkpoint_file)

    return np.array(all_embeddings)

# Ejecutamos la función con control de fallos
# Nota: Si se vuelve a cortar, solo vuelve a ejecutar esta celda y leerá el archivo .pkl automáticamente
embeddings_array = get_embeddings_en_lotes_con_retry(textos, batch_size=50) # Bajamos el batch_size a 50 para ser más estables

🔄 Checkpoint detectado. Reanudando desde el lote 0 (0 textos ya procesados).
✅ Lote 0 a 50 procesado correctamente.
✅ Lote 50 a 100 procesado correctamente.
✅ Lote 100 a 150 procesado correctamente.
✅ Lote 150 a 200 procesado correctamente.
✅ Lote 200 a 250 procesado correctamente.
✅ Lote 250 a 300 procesado correctamente.
✅ Lote 300 a 350 procesado correctamente.
✅ Lote 350 a 400 procesado correctamente.
✅ Lote 400 a 450 procesado correctamente.
✅ Lote 450 a 500 procesado correctamente.
✅ Lote 500 a 550 procesado correctamente.
✅ Lote 550 a 600 procesado correctamente.
✅ Lote 600 a 650 procesado correctamente.
✅ Lote 650 a 700 procesado correctamente.
✅ Lote 700 a 750 procesado correctamente.
✅ Lote 750 a 800 procesado correctamente.
✅ Lote 800 a 850 procesado correctamente.
✅ Lote 850 a 900 procesado correctamente.
✅ Lote 900 a 950 procesado correctamente.
✅ Lote 950 a 1000 procesado correctamente.
✅ Lote 1000 a 1050 procesado correctamente.
✅ Lote 1050 a 1100 procesado correctamente

In [ ]:
import pandas as pd
import pickle
import numpy as np

# 1. Volver a cargar el dataset y aplicar la limpieza (Paso 2 y 3 originales)
print("🔄 Cargando y limpiando el dataset original...")
df = pd.read_csv('healthcare_dataset.csv')
df = df.drop_duplicates()
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

string_cols = df.select_dtypes(include=['object']).columns
df[string_cols] = df[string_cols].fillna('Unknown')

for col in df.columns:
    if 'date' in col:
        df[col] = pd.to_datetime(df[col]).dt.strftime('%Y-%m-%d')

# Recreamos la función narrativa
def embedding_narrative(row):
    return (
        f"Paciente: {row['name']}. Edad: {row['age']}. Género: {row['gender']}. "
        f"Grupo Sanguíneo: {row['blood_type']}. Condición Médica: {row['medical_condition']}. "
        f"Fecha de Admisión: {row['date_of_admission']}. Doctor asignado: {row['doctor']}. "
        f"Hospital: {row['hospital']}. Aseguradora: {row['insurance_provider']}. "
        f"Monto de Facturación: ${row['billing_amount']:.2f}. Número de Habitación: {row['room_number']}. "
        f"Tipo de Admisión: {row['admission_type']}. Fecha de Alta: {row['discharge_date']}. "
        f"Medicamento recetado: {row['medication']}. Resultados de la prueba: {row['test_results']}."
    )

df['text_context'] = df.apply(embedding_narrative, axis=1)
textos_completos = df['text_context'].tolist()
print("✅ Variable 'df' y 'textos_completos' restauradas con éxito.")

# 2. Cargamos los embeddings guardados desde el archivo checkpoint
checkpoint_file = "embeddings_checkpoint.pkl"

try:
    with open(checkpoint_file, 'rb') as f:
        checkpoint_data = pickle.load(f)
        all_embeddings = checkpoint_data.get('embeddings', [])
        print(f"🔄 Archivo recuperado: Se detectaron {len(all_embeddings)} embeddings en el disco.")

        # 3. Realizamos el recorte seguro a los primeros 1,000 registros
        textos_limitados = textos_completos[:1000]
        embeddings_limitados = np.array(all_embeddings[:1000])

        print(f"\n🚀 ¡Todo listo para continuar!")
        print(f"-> Textos limitados listos: {len(textos_limitados)}")
        print(f"-> Matriz de embeddings limitada lista: {embeddings_limitados.shape}")

except FileNotFoundError:
    print(f"\n❌ Crítico: El archivo '{checkpoint_file}' no se encuentra en el almacenamiento local de Colab.")
    print("Si el entorno se borró por completo (icono de carpeta vacío), tendrás que procesar los primeros 1,000 desde cero.")

🔄 Cargando y limpiando el dataset original...
✅ Variable 'df' y 'textos_completos' restauradas con éxito.

❌ Crítico: El archivo 'embeddings_checkpoint.pkl' no se encuentra en el almacenamiento local de Colab.
Si el entorno se borró por completo (icono de carpeta vacío), tendrás que procesar los primeros 1,000 desde cero.


In [ ]:
nuevo

In [ ]:
import time
import numpy as np
from langchain_mistralai import MistralAIEmbeddings

# 1. Inicializar el modelo de embeddings de Mistral
embeddings = MistralAIEmbeddings(model="mistral-embed")

# 2. Seleccionar solo los primeros 1,000 textos
textos_limitados = textos_completos[:1000]

# 3. Procesar en lotes pequeños (ej. de 50 en 50) para asegurar estabilidad
embeddings_list = []
batch_size = 50
total = len(textos_limitados)

print(f"🚀 Iniciando el procesamiento de {total} registros desde cero...")

for i in range(0, total, batch_size):
    lote = textos_limitados[i:i + batch_size]
    try:
        # Generar embeddings para el lote actual
        lote_embeddings = embeddings.embed_documents(lote)
        embeddings_list.extend(lote_embeddings)
        print(f"✅ Procesados registros del {i} al {min(i + batch_size, total)}")
        time.sleep(0.5) # Pausa de cortesía para la API
    except Exception as e:
        print(f"❌ Error en el lote {i}: {e}")
        break

# Convertir la lista final en un array de NumPy
embeddings_limitados = np.array(embeddings_list)
print(f"\n✨ ¡Proceso completado! Matriz de embeddings lista con dimensiones: {embeddings_limitados.shape}")

ModuleNotFoundError: No module named 'langchain_mistralai'

In [ ]:
!pip install -q langchain-mistralai

In [ ]:
import time
import numpy as np
import os # Added import
from langchain_mistralai import MistralAIEmbeddings
from google.colab import userdata

# 1. Recuperar la API Key de Mistral
mistral_api_key = userdata.get('MISTRAL_API_KEY')

# Validar que la clave API exista
if not mistral_api_key:
    raise ValueError("MISTRAL_API_KEY no encontrada en los secretos de Colab. Por favor, configúrala para continuar.")

# 2. Establecer la clave API como variable de entorno
os.environ["MISTRAL_API_KEY"] = mistral_api_key

# 3. Inicializar el modelo de embeddings de Mistral (ya no se necesita pasar la clave directamente)
embeddings = MistralAIEmbeddings(model="mistral-embed")

# 4. Seleccionar solo los primeros 1,000 textos
textos_limitados = textos_completos[:1000]

# 5. Procesar en lotes pequeños (ej. de 50 en 50) para asegurar estabilidad
embeddings_list = []
batch_size = 50
total = len(textos_limitados)

print(f"🚀 Iniciando el procesamiento de {total} registros desde cero...")

for i in range(0, total, batch_size):
    lote = textos_limitados[i:i + batch_size]
    try:
        # Generar embeddings para el lote actual
        lote_embeddings = embeddings.embed_documents(lote)
        embeddings_list.extend(lote_embeddings)
        print(f"✅ Procesados registros del {i} al {min(i + batch_size, total)}")
        time.sleep(0.5) # Pausa de cortesía para la API
    except Exception as e:
        print(f"❌ Error en el lote {i}: {e}")
        break

# Convertir la lista final en un array de NumPy
embeddings_limitados = np.array(embeddings_list)
print(f"\n✨ ¡Proceso completado! Matriz de embeddings lista con dimensiones: {embeddings_limitados.shape}")

🚀 Iniciando el procesamiento de 1000 registros desde cero...
✅ Procesados registros del 0 al 50
✅ Procesados registros del 50 al 100
✅ Procesados registros del 100 al 150
✅ Procesados registros del 150 al 200
✅ Procesados registros del 200 al 250
✅ Procesados registros del 250 al 300
✅ Procesados registros del 300 al 350
✅ Procesados registros del 350 al 400
✅ Procesados registros del 400 al 450
✅ Procesados registros del 450 al 500
✅ Procesados registros del 500 al 550
✅ Procesados registros del 550 al 600
✅ Procesados registros del 600 al 650
✅ Procesados registros del 650 al 700
✅ Procesados registros del 700 al 750
✅ Procesados registros del 750 al 800
✅ Procesados registros del 800 al 850
✅ Procesados registros del 850 al 900
✅ Procesados registros del 900 al 950
✅ Procesados registros del 950 al 1000

✨ ¡Proceso completado! Matriz de embeddings lista con dimensiones: (1000, 1024)


In [ ]:
import sys
!{sys.executable} -m pip install -q  langchain-community

from langchain_community.vectorstores import FAISS

# Emparejar cada texto con su respectivo vector matemático
text_embeddings_pairs = list(zip(textos_limitados, embeddings_limitados))

# Crear la base de datos vectorial FAISS pasándole los vectores ya calculados
vector_store = FAISS.from_embeddings(
    text_embeddings=text_embeddings_pairs,
    embedding=embeddings,
    metadatas=[{"source": "healthcare_sample"} for _ in range(len(textos_limitados))]
)

# Configurar el recuperador (Retriever) para traer los 3 registros más parecidos por consulta
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

print("🎯 ¡Base de datos vectorial FAISS creada de forma óptima con 1,000 registros!")

ValueError: not enough values to unpack (expected 2, got 0)

In [6]:
# 0. Forzar la instalación de dependencias en el entorno actual de Colab
!pip install -q langchain-mistralai langchain-community faiss-cpu pandas numpy

import os
import time
import numpy as np
import pandas as pd
from google.colab import userdata
from langchain_mistralai import MistralAIEmbeddings
from langchain_community.vectorstores import FAISS

# --- REFUERZO DE SEGURIDAD PARA LA API KEY ---
try:
    os.environ["MISTRAL_API_KEY"] = userdata.get('MISTRAL_API_KEY')
    if not os.environ["MISTRAL_API_KEY"]:
        raise ValueError("La API Key está vacía.")
except Exception:
    # Si falla la lectura automatizada, te pedirá que la ingreses manualmente en consola para no trabarte
    import getpass
    print("⚠️ No se pudo leer 'MISTRAL_API_KEY' desde los Secretos de Colab.")
    os.environ["MISTRAL_API_KEY"] = getpass.getpass("Por favor, pega tu Mistral API Key aquí y presiona Enter: ")

# 1. Asegurar que los textos estén cargados en memoria
print("\n🔄 1/4 Restaurando textos desde el DataFrame...")
df = pd.read_csv('healthcare_dataset.csv')
df = df.drop_duplicates()
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df[df.select_dtypes(include=['object']).columns] = df.select_dtypes(include=['object']).columns.str.strip()

def embedding_narrative(row):
    return (
        f"Paciente: {row['name']}. Edad: {row['age']}. Género: {row['gender']}. "
        f"Grupo Sanguíneo: {row['blood_type']}. Condición Médica: {row['medical_condition']}. "
        f"Fecha de Admisión: {row['date_of_admission']}. Doctor asignado: {row['doctor']}. "
        f"Hospital: {row['hospital']}. Aseguradora: {row['insurance_provider']}. "
        f"Monto de Facturación: ${row['billing_amount']:.2f}. Número de Habitación: {row['room_number']}. "
        f"Tipo de Admisión: {row['admission_type']}. Fecha de Alta: {row['discharge_date']}. "
        f"Medicamento recetado: {row['medication']}. Resultados de la prueba: {row['test_results']}."
    )

df['text_context'] = df.apply(embedding_narrative, axis=1)
textos_limitados = df['text_context'].tolist()[:1000]
print(f"✅ Textos listos: {len(textos_limitados)} registros seleccionados.")

# 2. Inicializar embeddings de Mistral
embeddings = MistralAIEmbeddings(model="mistral-embed")

# 3. Generar los vectores en lotes controlados
print("\n🚀 2/4 Generando embeddings con la API de Mistral...")
embeddings_list = []
batch_size = 50
total = len(textos_limitados)

for i in range(0, total, batch_size):
    lote = textos_limitados[i:i + batch_size]
    try:
        lote_embeddings = embeddings.embed_documents(lote)
        embeddings_list.extend(lote_embeddings)
        print(f"   -> Procesados registros del {i} al {min(i + batch_size, total)}")
        time.sleep(0.5)
    except Exception as e:
        print(f"❌ Error crítico en lote {i}: {e}")
        break

embeddings_limitados = np.array(embeddings_list)

# 4. Crear el índice FAISS
from langchain_community.vectorstores import FAISS

print("🎯 3/4 Emparejando y creando la base de datos vectorial FAISS...")

if len(textos_limitados) == len(embeddings_limitados) and len(embeddings_limitados) > 0:
    # Creamos la lista de tuplas (texto, vector)
    text_embeddings_pairs = list(zip(textos_limitados, embeddings_limitados))

    # Corregido: Pasamos los datos de forma posicional para evitar conflictos de nombres de argumentos
    vector_store = FAISS.from_embeddings(
        text_embeddings_pairs, # 1er argumento posicional: las tuplas (texto, embedding)
        embeddings,            # 2do argumento posicional: el modelo de embeddings
        metadatas=[{"source": "healthcare_sample"} for _ in range(len(textos_limitados))]
    )

    # Creamos el recuperador global
    retriever = vector_store.as_retriever(search_kwargs={"k": 3})
    print("\n✨ 4/4 ¡Base de datos vectorial FAISS creada con éxito con 1,000 registros!")
else:
    print("\n❌ Error: La cantidad de textos y embeddings no coincide.")


🔄 1/4 Restaurando textos desde el DataFrame...
✅ Textos listos: 1000 registros seleccionados.

🚀 2/4 Generando embeddings con la API de Mistral...
   -> Procesados registros del 0 al 50
   -> Procesados registros del 50 al 100
   -> Procesados registros del 100 al 150
   -> Procesados registros del 150 al 200
   -> Procesados registros del 200 al 250
   -> Procesados registros del 250 al 300
   -> Procesados registros del 300 al 350
   -> Procesados registros del 350 al 400
   -> Procesados registros del 400 al 450
   -> Procesados registros del 450 al 500
   -> Procesados registros del 500 al 550
   -> Procesados registros del 550 al 600
   -> Procesados registros del 600 al 650
   -> Procesados registros del 650 al 700
   -> Procesados registros del 700 al 750
   -> Procesados registros del 750 al 800
   -> Procesados registros del 800 al 850
   -> Procesados registros del 850 al 900
   -> Procesados registros del 900 al 950
   -> Procesados registros del 950 al 1000
🎯 3/4 Emparejan

In [3]:
import os
from google.colab import userdata
from langchain_mistralai import ChatMistralAI
from langchain_core.prompts import ChatPromptTemplate

# 1. Asegurar la API Key de Mistral
if "MISTRAL_API_KEY" not in os.environ:
    os.environ["MISTRAL_API_KEY"] = userdata.get('MISTRAL_API_KEY')

# 2. Inicializar el modelo de Mistral para la generación
llm = ChatMistralAI(model="mistral-large-latest", temperature=0)

# 3. La condición médica a consultar
consulta_medica = "Diabetes"

print(f"🔍 Buscando registros directamente en la base de datos vectorial FAISS para: {consulta_medica}...\n")

# 4. Alternativa segura: Buscamos directamente usando 'vector_store' (o 'retriever' si existe)
try:
    if 'retriever' in locals() or 'retriever' in globals():
        docs_recuperados = retriever.vector_store.similarity_search(consulta_medica, k=5)
    else:
        # Si 'retriever' no se declaró, usamos 'vector_store' que es el objeto raíz de FAISS
        docs_recuperados = vector_store.similarity_search(consulta_medica, k=5)

    print(f"✅ Se recuperaron {len(docs_recuperados)} registros médicos relevantes.")

except NameError:
    print("❌ Error: Las variables de FAISS no están en memoria. Ejecuta la celda de arriba (Paso 3 y 4) donde procesaste los 1,000 registros antes de correr esta celda.")
    raise

# 5. Unir los documentos en un solo bloque de texto limpio
contexto_str = "\n\n".join([doc.page_content for doc in docs_recuperados])

# 6. Diseñar el Prompt estructurado para Mistral
system_prompt = (
    "Eres un experto profesional en análisis de datos de salud e inteligencia artificial.\n"
    "A continuación, se te proporcionará un fragmento de la base de conocimientos del hospital "
    "con registros reales de pacientes.\n\n"
    "CONTEXTO RECUPERADO:\n{contexto}\n\n"
    "Tu tarea es analizar estrictamente este contexto y extraer exactamente 5 conclusiones claras, "
    "numéricas o descriptivas sobre los tratamientos, medicamentos, costos de facturación o "
    "patrones que encuentres en estos pacientes específicos. No inventes información fuera del contexto."
)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("user", "Basándote en los datos anteriores, genera las 5 conclusiones sobre los pacientes con {consulta}.")
])

# 7. Formatear el prompt con el contexto real del dataset
prompt_final = prompt_template.format_messages(contexto=contexto_str, consulta=consulta_medica)

print("🤖 Enviando contexto optimizado a Mistral AI...")

# 8. Obtener el reporte final
respuesta = llm.invoke(prompt_final)

print("\n--- RESPUESTA FINAL DEL SISTEMA RAG ---")
print(respuesta.content)


🔍 Buscando registros directamente en la base de datos vectorial FAISS para: Diabetes...

❌ Error: Las variables de FAISS no están en memoria. Ejecuta la celda de arriba (Paso 3 y 4) donde procesaste los 1,000 registros antes de correr esta celda.


NameError: name 'vector_store' is not defined

In [4]:
 %whos

Variable             Type              Data/Info
------------------------------------------------
ChatMistralAI        ModelMetaclass    <class 'langchain_mistral<...>at_models.ChatMistralAI'>
ChatPromptTemplate   ModelMetaclass    <class 'langchain_core.pr<...>chat.ChatPromptTemplate'>
consulta_medica      str               Diabetes
llm                  ChatMistralAI     metadata={'versions': {'l<...>ature=0.0 model_kwargs={}
os                   module            <module 'os' (frozen)>
userdata             module            <module 'google.colab.use<...>oogle/colab/userdata.py'>


In [5]:
# 0. Instalación forzada de dependencias necesarias
!pip install -q langchain-mistralai langchain-community faiss-cpu pandas numpy

import os
import time
import numpy as np
import pandas as pd
from google.colab import userdata
from langchain_mistralai import MistralAIEmbeddings, ChatMistralAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate

# 1. Configuración e inicio seguro de la API Key de Mistral
if "MISTRAL_API_KEY" not in os.environ:
    os.environ["MISTRAL_API_KEY"] = userdata.get('MISTRAL_API_KEY')

# 2. Re-generación controlada de los 1,000 textos narrativos
print("🔄 1/5 Cargando dataset y preparando registros médicos...")
try:
    df = pd.read_csv('healthcare_dataset.csv')
    df = df.drop_duplicates()
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    df[df.select_dtypes(include=['object']).columns] = df.select_dtypes(include=['object']).columns.str.strip()
except FileNotFoundError:
    raise FileNotFoundError("⚠️ Crítico: Asegúrate de volver a subir 'healthcare_dataset.csv' a la barra lateral izquierda de Colab.")

def embedding_narrative(row):
    return (
        f"Paciente: {row['name']}. Edad: {row['age']}. Género: {row['gender']}. "
        f"Grupo Sanguíneo: {row['blood_type']}. Condición Médica: {row['medical_condition']}. "
        f"Fecha de Admisión: {row['date_of_admission']}. Doctor asignado: {row['doctor']}. "
        f"Hospital: {row['hospital']}. Aseguradora: {row['insurance_provider']}. "
        f"Monto de Facturación: ${row['billing_amount']:.2f}. Número de Habitación: {row['room_number']}. "
        f"Tipo de Admisión: {row['admission_type']}. Fecha de Alta: {row['discharge_date']}. "
        f"Medicamento recetado: {row['medication']}. Resultados de la prueba: {row['test_results']}."
    )

df['text_context'] = df.apply(embedding_narrative, axis=1)
textos_limitados = df['text_context'].tolist()[:1000]
print(f"✅ Subconjunto listo: {len(textos_limitados)} registros seleccionados para mitigar sobrecarga.")

# 3. Generación instantánea de Embeddings con Mistral (Lotes de 50)
print("\n🚀 2/5 Solicitando vectores matemáticos a Mistral API...")
embeddings = MistralAIEmbeddings(model="mistral-embed")
embeddings_list = []
batch_size = 50
total = len(textos_limitados)

for i in range(0, total, batch_size):
    lote = textos_limitados[i:i + batch_size]
    try:
        lote_embeddings = embeddings.embed_documents(lote)
        embeddings_list.extend(lote_embeddings)
        print(f"   -> Lote {i // batch_size + 1}: Procesados registros del {i} al {min(i + batch_size, total)}")
        time.sleep(0.4)
    except Exception as e:
        print(f"❌ Error en lote {i}: {e}")
        break

embeddings_limitados = np.array(embeddings_list)

# 4. Creación e Indexación en Base de Datos Vectorial FAISS
print("\n🎯 3/5 Construyendo índice de similitud FAISS...")
text_embeddings_pairs = list(zip(textos_limitados, embeddings_limitados))
vector_store = FAISS.from_embeddings(text_embeddings_pairs, embeddings)
print("✅ Índice FAISS instanciado correctamente en memoria activa.")

# 5. Recuperación RAG (Búsqueda de Similitud Avanzada)
consulta_medica = "Diabetes"
print(f"\n🔍 4/5 Buscando evidencias en FAISS para la condición: '{consulta_medica}'...")
docs_recuperados = vector_store.similarity_search(consulta_medica, k=5)
contexto_str = "\n\n".join([doc.page_content for doc in docs_recuperados])
print(f"✅ Contexto clínico relevante recuperado ({len(docs_recuperados)} historias de pacientes).")

# 6. Generación Aumentada con Mistral Large
print("\n🤖 5/5 Sintetizando conclusiones ejecutivas con Mistral Large...")
llm = ChatMistralAI(model="mistral-large-latest", temperature=0)

system_prompt = (
    "Eres un experto profesional en análisis de datos de salud e inteligencia artificial.\n"
    "A continuación, se te proporcionará un fragmento de la base de conocimientos del hospital "
    "con registros reales de pacientes obtenidos por RAG.\n\n"
    "CONTEXTO RECUPERADO:\n{contexto}\n\n"
    "Tu tarea es analizar estrictamente este contexto y extraer exactamente 5 conclusiones claras, "
    "numéricas o descriptivas sobre los tratamientos, medicamentos o costos de facturación que encuentres "
    "en estos pacientes específicos. Sé conciso y profesional."
)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("user", "Basándote en los datos recuperados, genera el reporte de 5 conclusiones sobre pacientes con {consulta}.")
])

prompt_final = prompt_template.format_messages(contexto=contexto_str, consulta=consulta_medica)
respuesta = llm.invoke(prompt_final)

print("\n" + "="*50)
print("✨ RESPUESTA FINAL DEL SISTEMA RAG OPTIMIZADO ✨")
print("="*50)
print(respuesta.content)

/tmp/ipykernel_11605/4128118008.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


🔄 1/5 Cargando dataset y preparando registros médicos...
✅ Subconjunto listo: 1000 registros seleccionados para mitigar sobrecarga.

🚀 2/5 Solicitando vectores matemáticos a Mistral API...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


   -> Lote 1: Procesados registros del 0 al 50
   -> Lote 2: Procesados registros del 50 al 100
   -> Lote 3: Procesados registros del 100 al 150
   -> Lote 4: Procesados registros del 150 al 200
   -> Lote 5: Procesados registros del 200 al 250
   -> Lote 6: Procesados registros del 250 al 300
   -> Lote 7: Procesados registros del 300 al 350
   -> Lote 8: Procesados registros del 350 al 400
   -> Lote 9: Procesados registros del 400 al 450
   -> Lote 10: Procesados registros del 450 al 500
   -> Lote 11: Procesados registros del 500 al 550
   -> Lote 12: Procesados registros del 550 al 600
   -> Lote 13: Procesados registros del 600 al 650
   -> Lote 14: Procesados registros del 650 al 700
   -> Lote 15: Procesados registros del 700 al 750
   -> Lote 16: Procesados registros del 750 al 800
   -> Lote 17: Procesados registros del 800 al 850
   -> Lote 18: Procesados registros del 850 al 900
   -> Lote 19: Procesados registros del 900 al 950
   -> Lote 20: Procesados registros del 950 